In [1]:
#!/usr/bin/env python3
"""
Batch HTML to Clean Text Converter using Docling
Processes all HTML files in a directory and outputs structured, readable content.
"""

import json
import re
import shutil
import sys
from pathlib import Path

from bs4 import BeautifulSoup
from docling.chunking import HierarchicalChunker
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter
from tqdm import tqdm

In [2]:
def setup_converter() -> DocumentConverter:
    """Initialize the Docling document converter with HTML support."""
    return DocumentConverter(
        allowed_formats=["html"],  # Only process HTML files
    )


converter = setup_converter()

In [3]:
def clean_html_file(html_content: Path) -> Path:
    """
    Clean HTML file using BeautifulSoup4.
    Removes scripts, styles, unnecessary attributes, and normalizes structure.
    Returns path to cleaned HTML file (temporary or overwritten).
    """

    # Parse with BeautifulSoup
    soup = BeautifulSoup(html_content, "lxml")  # or 'html.parser'

    # Specific data-testid values to remove
    unwanted_testids = [
        "table-mobile",
        "table-of-content",
        "recommendations-container",
        "other-categories",
    ]

    for testid in unwanted_testids:
        for element in soup.find_all(attrs={"data-testid": testid}):
            element.decompose()
    for element in soup.find_all(attrs={"data-schema-path": "button"}):
        element.decompose()
    for element in soup.find_all(attrs={"id": "form"}):
        element.decompose()

    # Pretty print the HTML
    return soup.prettify()

In [4]:
def build_chunks(document, *, chunker):
    chunks = chunker.chunk(document)
    chunk_list = []
    for chunk in chunks:
        rich_text = chunker.contextualize(chunk)
        chunk_list.append(remove_markdown_links(rich_text))
    return chunk_list


def remove_markdown_links(text: str) -> str:
    # This regex removes markdown links [text](url) and leaves just the 'text'
    return re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", text)

In [5]:
def process_html_file(
    converter: DocumentConverter, file_path: Path, output_dir: Path
) -> dict | None:
    """
    Process a single HTML file and save outputs in multiple formats.

    Returns a dictionary with file info and conversion status.
    """
    result = {"file": str(file_path), "success": False, "error": None}

    try:
        base_name = file_path.stem
        # Create output files for this specific HTML file
        file_output_dir = output_dir / base_name
        ensure_empty_dir(file_output_dir)

        # Convert the HTML file
        with file_path.open("r", encoding="utf-8") as f:
            raw_html = f.read()
        clean_html = clean_html_file(raw_html)
        conv_result = converter.convert_string(
            clean_html, format=InputFormat.HTML, name=base_name
        )

        # 1. Export to Markdown - good for LLM consumption and readability
        md_path = file_output_dir / "index.md"
        conv_result.document.save_as_markdown(md_path)

        # 2. Export to HTML - clean, stripped-down HTML (Docling v2+ feature)
        html_path = file_output_dir / "index.html"
        conv_result.document.save_as_html(html_path)

        # 3. Export to JSON - lossless format for programmatic analysis
        json_path = file_output_dir / "index.json"
        conv_result.document.save_as_json(json_path)

        # chunker = HybridChunker()
        chunker = HierarchicalChunker()
        chunk_list = build_chunks(conv_result.document, chunker=chunker)
        chunks_path = file_output_dir / "chunks.json"
        result["chunks"] = chunk_list
        with chunks_path.open("w", encoding="utf-8") as f:
            f.write(json.dumps(chunk_list))

        result["success"] = True
        return result

    except Exception as e:
        result["error"] = str(e)
        return result


def ensure_empty_dir(directory_path: Path) -> None:
    """
    Ensure a directory is empty by removing and recreating it.
    """
    if directory_path.exists():
        shutil.rmtree(directory_path)  # Delete entire directory tree

    directory_path.mkdir(parents=True, exist_ok=True)

In [6]:
process_html_file(
    converter,
    Path(
        "../data/raw/tbank_help_pages/bank_help_debit-cards_tinkoff-black-premium_earn-with-card_cashback-intro.html"
    ),
    Path("../data/interim/docling"),
)

{'file': '../data/raw/tbank_help_pages/bank_help_debit-cards_tinkoff-black-premium_earn-with-card_cashback-intro.html',
 'success': True,
 'error': None,
 'chunks': ['Как работает кэшбэк\nЧто такое кэшбэк и откуда он берется?\nКэшбэк - это часть денег, которую мы возвращаем клиенту за то, что он оплатил товары или услуги картой Т-Банка. Кэшбэк рассчитывается в процентах от стоимости покупки и зачисляется на карту. По карте Black Premium кэшбэк начисляем в рублях - его можно потратить на что угодно.',
  'Как работает кэшбэк\nЧто такое кэшбэк и откуда он берется?\nВот как работает кэшбэк:',
  'Как работает кэшбэк\nЧто такое кэшбэк и откуда он берется?\n1. Вы оплатили покупку картой Black Premium.\n2. Магазин заплатил комиссию банку, в котором обслуживается.\n3. Банк магазина заплатил часть комиссии Т-Банку, выпустившему вашу карту.\n4. Т-Банк поделился с вами частью этой суммы.',
  'Как работает кэшбэк\nКакие виды кэшбэка есть по карте Black Premium?\nДо 15% за покупки в четырех категори

In [7]:
def process_all(input_dir: str, output_dir: str):
    """Main execution function."""
    # Configuration
    output_path = Path(output_dir)
    input_path = Path(input_dir)

    if not input_path.exists() or not input_path.is_dir():
        print(f"❌ Error: Directory '{input_dir}' does not exist.")
        sys.exit(1)

    # Find all HTML files
    html_files = list(input_path.glob("*.html"))

    if not html_files:
        print(f"⚠️  No HTML files found in '{input_dir}'")
        sys.exit(0)

    print(f"\n📁 Found {len(html_files)} HTML file(s) in '{input_dir}'")
    print(f"📂 Output directory: {output_path.absolute()}\n")

    # Create output directory
    output_path.mkdir(parents=True, exist_ok=True)

    # Process each file
    results = []
    for html_file in tqdm(html_files):
        result = process_html_file(converter, html_file, output_path)
        results.append(result)

        if not result["success"]:
            print(f"    ❌ Failed: {result['error']}")

    # Print final summary
    successful = sum(1 for r in results if r["success"])

    print("✨ Processing complete!")
    print(f"   ✅ Successful: {successful}/{len(html_files)}")
    print(f"   📁 Output directory: {output_path.absolute()}")

In [8]:
process_all("../data/raw/tbank_help_pages", "../data/interim/docling")


📁 Found 1395 HTML file(s) in '../data/raw/tbank_help_pages'
📂 Output directory: /Users/vladimirklepov/Documents/rag-smith/notebooks/../data/interim/docling



100%|█████████████████████████████████████████████████████████████████████| 1395/1395 [01:56<00:00, 12.00it/s]

✨ Processing complete!
   ✅ Successful: 1395/1395
   📁 Output directory: /Users/vladimirklepov/Documents/rag-smith/notebooks/../data/interim/docling
